# Weight Gradient Analysis

Which weights matter most for specific predictions:
sensitivity profiles, per-layer and per-component gradients.

## Why This Matters

Gradient weight traces how training signals and attribution scores flow through the network. Understanding gradient structure reveals which components are most trainable, which carry the strongest attribution signals, and how LayerNorm affects learning.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_gradient_analysis import (
    weight_sensitivity_profile, layer_weight_gradient_norms,
    attention_weight_gradients, mlp_weight_gradients,
    weight_gradient_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Weight Sensitivity Profile

Top 10 most sensitive parameters.

In [ ]:
result = weight_sensitivity_profile(model, tokens)
print(f"Total parameters: {result['n_params']}")
print(f"Mean grad norm: {result['mean_grad_norm']:.6f}")
for p in result['top_sensitive'][:5]:
    print(f"  Param {p['param_index']}: grad={p['grad_norm']:.6f}, "
          f"rel_sensitivity={p['relative_sensitivity']:.6f}")

## Layer Gradient Norms

Which layers have the strongest gradients?

In [ ]:
result = layer_weight_gradient_norms(model, tokens)
print(f"Most sensitive: Layer {result['most_sensitive_layer']}")
for p in result['per_layer']:
    bar = '█' * int(p['fraction'] * 40)
    print(f"  Layer {p['layer']}: norm={p['grad_norm']:.6f}, frac={p['fraction']:.2f} {bar}")

## Attention Weight Gradients

Q, K, V, O gradient comparison per layer.

In [ ]:
for layer in range(4):
    result = attention_weight_gradients(model, tokens, layer=layer)
    print(f"  Layer {layer}: dominant={result['dominant_matrix']}")
    for name, norm in result['grad_norms'].items():
        bar = '█' * int(result['fractions'][name] * 40)
        print(f"    {name}: {norm:.6f} ({result['fractions'][name]:.2f}) {bar}")

## MLP Weight Gradients

In [ ]:
for layer in range(4):
    result = mlp_weight_gradients(model, tokens, layer=layer)
    print(f"  Layer {layer}: dominant={result['dominant_matrix']}")
    for name, norm in result['grad_norms'].items():
        print(f"    {name}: {norm:.6f} ({result['fractions'][name]:.2f})")

## Gradient Summary

In [ ]:
result = weight_gradient_summary(model, tokens)
print(f"Most sensitive layer: {result['most_sensitive_layer']}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: grad_norm={p['grad_norm']:.6f}, frac={p['fraction']:.2f}")